# 모방학습으로 시작해서 강화학습으로 넘어서기STEVAL-EDUKIT01 로터리 역진자. **하나의 정책 네트워크**를 세 단계로 학습시킨다.$$\text{전문가 시연} \xrightarrow{\;\text{BC}\;} \pi_0 \xrightarrow{\;\text{DAgger}\;} \pi_1 \xrightarrow{\;\text{SAC}\;} \pi^\*$$## 왜 융합하는가두 방법은 서로의 약점에 정확히 맞물린다.| | 할 수 있는 것 | 못 하는 것 ||---|---|---|| **강화학습** | 보상을 직접 최대화 → **전문가보다 잘할 수 있다** | 어려운 과제를 **우연히** 성공해야 시작된다 || **모방학습** | 첫 에피소드부터 올바른 행동 | 베낀 것보다 **잘할 수 없다** |`swingup` 이 그 "어려운 과제"다. 보상의 수직 항을 한 번이라도 받으려면매달린 막대를 ±180° 넘겨 **느리게** 도착시켜야 하고, 그걸 **팔 140° 예산 안에서** 해야 한다.무작위 탐험이 이걸 우연히 할 확률은 사실상 0이고, 그러면 **critic은 평가해야 할 행동을 본 적이 없다.**복제는 그 행동을 **첫 판에** 만들어 준다. 그리고 복제는 거기까지다.그래서 복제한 정책을 **출발점**으로 삼고 RL이 그 위에서 개선한다.## 성립 조건: 같은 언어하나의 네트워크를 물려주려면 모든 단계가 같은 관측·행동 공간을 써야 한다.기존 두 폴더는 달랐다.| | `behavior_cloning_yeen` | `rl_yeen` ||---|---|---|| 관측 | `[sin θ, cos θ, θ', φ, φ']` × K프레임 | `[cos θ, sin θ, θ', φ, φ']` × 1 || 행동 | `u` [rad/s²] | `a ∈ [-1, 1]` (tanh) || 주기 | 100 Hz (`rm.Loop`) | 50 Hz (자체 RK4) |그래서 **모방 단계를 RL 규약으로 다시 한다.** 이게 이 폴더가 하는 일이다.## 망가지기 쉬운 세 지점복제한 가중치는 SAC의 기본 설정에서 **수백 스텝 안에 지워진다.** 세 가지를 막아야 한다.1. **critic이 난수다 → 그 기울기도 난수다.** 정책을 움직이기 전에 시연만으로 critic을 먼저 2,000스텝 학습한다 (`critic_warm`).2. **엔트로피 목표가 정책을 무작위로 끌어당긴다.** `log_std` 를 작게(≈0.1) 두고 `alpha` 를 0.02에서 시작한다. 보상이 정당화하면 알아서 올라간다.3. **균일 난수 탐험 구간(`start_steps`)을 없앤다.** 그건 cold start 가 필요한 것이고 warm start 가 반드시 피해야 하는 것이다.그리고 시연으로 끌어당기는 항을 **감쇠**시켜 결국 떼어낸다.$$\mathcal{L}_{\text{actor}} \;=\; \alpha\log\pi \;-\; \min(Q_1,Q_2) \;+\; \underbrace{c(t)}_{\;1 \to 0\;} \big\|\tanh\mu(o) - a^\*\big\|^2$$$c(t)$ 가 0이 되면 정책은 전문가를 떠날 수 있다. **그때 전문가를 넘어서는지가 결과다.**## 측정의미 있는 비교는 **같은 예산의 cold start** 하나뿐이다. "SAC가 풀었다" 는 출발점이 없던 실행과나란히 놓이지 않으면 아무 말도 아니다 (`--baseline`).채점은 두 곳에서 한다.- **학습 환경** (50 Hz, 완벽한 센싱) — SAC가 실제로 최적화한 숫자- **채점장** `arena.py` (100 Hz `rm.Loop`, 양자화 엔코더, 4 ms 지연) — **실기의 루프**왼쪽에서만 이기는 단계는 전이에 실패한 것이다.

## 1절 — 준비

In [ ]:
import os, sys, subprocess, pathlibREPO = pathlib.Path("/content/Pendulum_control")if not REPO.exists():    subprocess.run(["git", "clone", "--depth", "1",                    "https://github.com/jooyongsim/Pendulum_control.git", str(REPO)], check=True)else:    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=False)os.chdir(REPO)PKGS = ("simulator", "behavior_cloning_yeen", "rl_yeen", "pipeline_yeen")for p in PKGS:    sys.path.insert(0, str(REPO / p))os.environ["PYTHONPATH"] = os.pathsep.join(str(REPO / p) for p in PKGS)os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"import numpy as np, pandas as pd, matplotlib, torchmatplotlib.rcParams["figure.dpi"] = 110print("torch", torch.__version__, "| cuda", torch.cuda.is_available())print("cwd  ", os.getcwd())

In [ ]:
import arena, animate, fusionimport rotary_model as rmimport rl_common as rlfrom eval_rl import teacherprint("plant   b =", rm.B_U, "| ACCEL_MAX =", round(rm.ACCEL_MAX, 2), "rad/s^2")print("env     obs", rl.OBS_NAMES, "| a in [-1,1] -> u = a *", round(rl.U_MAX, 1))print("        50 Hz agent step, 1 kHz RK4 physics, arm stop", round(np.degrees(rl.ARM_LIM)), "deg")print("arena   100 Hz rm.Loop, quantised encoder, 4 ms delay")print("fusion  LOG_STD0", fusion.LOG_STD0, "| ALPHA0", fusion.ALPHA0)

## 2절 — 문제를 먼저 본다: 우연히 성공할 수 있는가보상의 수직 항을 받으려면 막대가 올라와야 한다. **무작위 정책으로 몇 번 되는가**를 센다.

In [ ]:
import numpy as np, math, pandas as pdrng = np.random.default_rng(0)rand_pol = lambda obs: float(rng.uniform(-1, 1))rows, touched = [], []for task in ("balance", "swingup"):    n_ep, n_up, rets, best_up = 30, 0, [], 0.0    for ep in range(n_ep):        lg = rl.rollout(rand_pol, task=task, reward="full", seed=ep)        up = float(np.mean(np.abs(np.degrees(lg["th"])) < rl.UPRIGHT_DEG))        if up > 0.01:            n_up += 1            # 시작 각도를 같이 본다: 올려서 세운 것인지, 처음부터 위에 있었던 것인지            touched.append(dict(task=task, ep=ep, theta0_deg=round(np.degrees(lg["th"][0]), 1),                                upright_pct=round(100 * up, 2)))        best_up = max(best_up, up)        rets.append(lg["r"].sum())    rows.append(dict(task=task, episodes=n_ep, ep_touching_upright=n_up,                     best_upright_frac=round(100 * best_up, 2), mean_return=round(np.mean(rets), 1)))display(pd.DataFrame(rows))print("수직을 스친 에피소드의 시작 각도 (|theta0| < 180 이면 매달린 상태가 아니었다는 뜻):")display(pd.DataFrame(touched))print("balance 는 이미 세워서 넘겨주므로 우연히도 수직 근처에 있다.")print("swingup 은 30판 중 거의 전부가 수직을 스치지조차 못하고, 스친 판은 초기 각도가")print("무작위로 위쪽이었던 경우다 -- '매달린 막대를 올려 세운' 판은 하나도 없다.")print("-> critic 이 평가해야 할 행동을 본 적이 없다. 이것이 cold start 가 막히는 지점이다.")

In [ ]:
# 전문가는 같은 과제에서 이렇게 한다rows = []for task in ("balance", "swingup"):    ev = rl.evaluate(teacher, task, "full")    rows.append(dict(task=task, **{k: round(ev[k], 2) for k in                     ("ret", "hold10", "hold5", "hold2", "bias_deg", "arm_max_deg", "failures")}))display(pd.DataFrame(rows))fig = animate.traces([("expert", arena.expert())], task="swingup")fig

## 3절 — 배관 점검 (1분)네 단계를 아주 작은 숫자로 전부 한 번 통과시킨다. **숫자는 무의미하고, 안 터지는지만 본다.**긴 학습을 걸기 전에 반드시 여기서 확인할 것.

In [ ]:
!python pipeline_yeen/fusion.py --task swingup --smoke

## 4절 — 단계별로 손으로 (무엇이 왜 바뀌는지 보려면)`fusion.py` 가 전부 해 주지만, 각 단계가 무엇을 하는지는 따로 돌려 봐야 보인다.### 4-1. 시연 수집전문가가 운전하고 모든 틱의 `(관측, 행동)` 을 기록한다. 행동에 작은 잡음을 섞는다 —완벽하게 똑같은 궤적만 모으면 그 궤적을 1 cm 벗어난 상태는 데이터에 없다.

In [ ]:
demos = fusion.collect(40, task="swingup", beta=1.0, noise=0.05, seed=1)print(demos["stats"])print("saturated |a*| > 0.99 :",      round(100 * float((abs(demos['a_star']) > 0.99).mean()), 1), "%")print("  -> 전문가는 상당 시간 포화 명령을 낸다. 그래서 BC 목표를 atanh(a*) 로 두면 발산한다.")print("     tanh 를 통과시킨 뒤 MSE 를 잰다:  loss = || tanh(mu(o)) - a* ||^2")

### 4-2. 행동 복제 — SAC actor의 **평균 머리**를 채운다`train_sac.Actor` 를 그대로 만들고, 그 `mu` 출력만 전문가 행동에 맞춘다.이 네트워크가 끝까지 간다 — 버리고 새로 만들지 않는다.

In [ ]:
import train_sac as tsimport pandas as pdactor = ts.Actor((256, 256))hist = fusion.bc_fit(actor, demos, epochs=60, seed=0)ev = rl.evaluate(lambda o: actor.act(o, True), "swingup", "full")print("\nBC 직후:", {k: round(ev[k], 2) for k in      ("ret", "hold10", "hold2", "bias_deg", "arm_max_deg", "failures")})print("teacher :", {k: round(v, 2) for k, v in rl.evaluate(teacher, 'swingup', 'full').items()                    if k in ("ret", "hold10", "hold2", "bias_deg", "arm_max_deg", "failures")})h = pd.DataFrame(hist)ax = h.plot(x="epoch", y=["train", "val"], logy=True, figsize=(7, 3),            title="behaviour cloning loss (MSE on a in [-1,1])")ax.set_ylabel("MSE")

In [ ]:
# 복제한 정책 한 판bc_ctrl = lambda o: actor.act(o, True)lg = rl.rollout(bc_ctrl, task="swingup", reward="full", theta0=3.14159)m = rl.metrics(lg)print({k: round(v, 2) for k, v in m.items() if isinstance(v, float)})import matplotlib.pyplot as pltfig, (a1, a2) = plt.subplots(2, 1, figsize=(9, 4.5), sharex=True)for lbl, f in (("expert", teacher), ("bc", bc_ctrl)):    g = rl.rollout(f, task="swingup", reward="full", theta0=3.14159)    a1.plot(g["t"], np.degrees(g["th"]), lw=1.2, label=lbl)    a2.plot(g["t"], np.degrees(g["phi"]), lw=1.2, label=lbl)a1.axhline(10, color="0.8", ls="--", lw=1); a1.axhline(-10, color="0.8", ls="--", lw=1)a2.axhline(140, color="#d62728", ls="--", lw=1); a2.axhline(-140, color="#d62728", ls="--", lw=1)a1.set_ylabel("theta [deg]"); a2.set_ylabel("phi [deg]"); a2.set_xlabel("t [s]")a1.legend(fontsize=8); a1.set_title("expert vs cloned policy", fontsize=10)fig.tight_layout()

### 4-3. DAgger — 학생이 가는 곳에 라벨을 붙인다BC의 문제는 **공변량 이동**이다. 학생은 전문가가 방문한 상태에서만 배웠는데,배포되면 자기 실수가 만든 상태로 들어간다.$$D_{i+1} = D_i \cup \{(z,\, a^\*(z)) : z \sim \pi_i \text{가 실제로 방문한 상태}\}$$여기서 중요한 구분이 하나 있다 — 한 번의 롤아웃에서 **두 가지**가 나온다.| | 무엇 | 어디로 ||---|---|---|| `a_exec` | 실제로 적용된 행동 | **리플레이 버퍼** (그것이 만든 보상·다음 상태와 함께) || `a_star` | 전문가가 여기서 할 행동 | **BC 데이터셋** (라벨) |이걸 섞는 것이 DAgger 구현의 흔한 버그다.

In [ ]:
demos, dlog = fusion.dagger(actor, demos, task="swingup", iters=3, episodes=20, epochs=30, seed=7)import pandas as pdpd.DataFrame(dlog).round(2)

### 4-4. SAC fine-tune — 여기서부터 전문가를 떠날 수 있다`bc_coef` 가 20,000 스텝에 걸쳐 1 → 0 으로 떨어진다. 그 뒤로 정책을 붙잡는 것은 보상뿐이다.

In [ ]:
import copywarm_actor = copy.deepcopy(actor)warm_actor, rep = fusion.finetune(warm_actor, demos, task="swingup", steps=60000,                                  bc_coef=1.0, bc_decay=20000, critic_warm=2000, seed=0)import pandas as pdcur = pd.DataFrame(rep["curve"])display(cur.round(2))ax = cur.plot(x="step", y=["ret", "train_ret"], figsize=(7, 3),              title="SAC fine-tune from the imitated policy")ax.axhline(rl.evaluate(teacher, "swingup", "full")["ret"], color="#d62728", ls="--", lw=1)ax.annotate("teacher", xy=(0.02, 0.9), xycoords="axes fraction", color="#d62728", fontsize=8)ax.set_ylabel("return")

## 5절 — 전부 한 번에 + cold 대조군여기가 본 실행이다. **같은 예산의 cold start** 를 함께 돌린다 — 그 대조가 결과다.`swingup`, 60,000 스텝 × 2 (CPU 40~70분 / GPU 더 빠름).

In [ ]:
!python pipeline_yeen/fusion.py --task swingup --demo-episodes 40 --bc-epochs 60 \    --dagger-iters 3 --dagger-episodes 20 --steps 60000 --baseline \    --csv pipeline_yeen/fusion_swingup.csv

In [ ]:
import pandas as pddf = pd.read_csv("pipeline_yeen/fusion_swingup.csv")order = ["teacher", "bc_swingup", "dagger_swingup", "sac_warm_swingup", "sac_cold_swingup"]df["stage"] = pd.Categorical(df["stage"], order, ordered=True)df = df.sort_values("stage").set_index("stage")df[["ret", "hold2", "bias", "arm", "fail", "arena_ret", "arena_hold2", "arena_stops"]] \  .style.format("{:.1f}").background_gradient(subset=["ret", "arena_ret"], cmap="Greens") \  .set_caption("왼쪽 4열 = 50 Hz 학습 환경 / arena_* = 100 Hz 실기 루프")

In [ ]:
# 리턴만 막대로import matplotlib.pyplot as pltfig, ax = plt.subplots(figsize=(7, 3))ax.bar(range(len(df)), df["ret"], color=["0.5", "#1f77b4", "#1f77b4", "#2ca02c", "#d62728"])ax.set_xticks(range(len(df)))ax.set_xticklabels([s.replace("_swingup", "") for s in df.index], fontsize=9)ax.axhline(df.loc["teacher", "ret"], color="0.3", ls="--", lw=1)ax.set_ylabel("return (50 Hz env)")ax.set_title("imitation gives the start, reinforcement passes the teacher", fontsize=10)fig.tight_layout()

In [ ]:
# 네 정책을 같은 시작에서 나란히from IPython.display import HTMLimport rl_common as rldef ctrl_of(name):    return rl.NumpyActor(f"pipeline_yeen/checkpoints/{name}.npz").as_rig_controller()entries = [("expert", arena.expert())]for n in ("bc_swingup", "dagger_swingup", "sac_warm_swingup", "sac_cold_swingup"):    import os    if os.path.exists(f"pipeline_yeen/checkpoints/{n}.npz"):        c = ctrl_of(n); c.label = n        entries.append((n.replace("_swingup", ""), c))ani, sc = animate.compare(entries, task="swingup", title="same start, 100 Hz rig loop")HTML(ani.to_jshtml())

In [ ]:
for k, v in sc.items():    print(f"{k:<12s} hold2 {v['hold2']:5.1f}%  bias {v['bias_deg']:+7.2f} deg  "          f"t_up {v['t_up']:5.2f} s  arm {v['arm_max_deg']:6.1f} deg  {v['stop'] or 'ok'}")

## 6절 — 모델 오차와 엔코더 영점앞 연구에서 배운 것: **깨끗한 조건은 아무것도 구분하지 못한다.**`offset 2` 는 엔코더 영점이 2° 틀린 조건이고, 전문가는 숨은 적분기 `SwingUp.b_hat` 으로 흡수한다.복제된 정책에는 그 적분기가 없다 — 8.9° 편향 사건의 정체가 이것이다.RL 쪽에도 적분기는 없다. **보상이 그 역할을 대신할 수 있는가**가 질문이다.

In [ ]:
import pandas as pdrows = []for lbl, c in entries:    for cond in arena.SUITES["robust"]:        plant = {k: v for k, v in cond.items() if k != "label"}        ev = arena.evaluate(c, "swingup", "full", **plant)        rows.append(dict(agent=lbl, cond=cond["label"], hold2=ev["hold2"],                         bias=ev["bias_deg"], arm=ev["arm_max_deg"], stops=ev["stops"]))t = pd.DataFrame(rows)piv = t.pivot(index="agent", columns="cond", values="hold2")piv = piv[[c["label"] for c in arena.SUITES["robust"]]]piv.style.format("{:.1f}").background_gradient(cmap="RdYlGn", vmin=0, vmax=100) \   .set_caption("hold2 [%] — 마지막 5초를 ±2° 안에서 유지한 비율")

In [ ]:
ani2, sc2 = animate.compare(entries, task="swingup", offset_deg=2.0,                            title="encoder zero off by 2 deg")from IPython.display import HTMLHTML(ani2.to_jshtml())

## 7절 — 정리| 단계 | 무엇을 최소/최대화 | 필요한 것 | 한계 ||---|---|---|---|| BC | 전문가 행동과의 차이 | 전문가 시연 한 번 | 자기 실수가 만든 상태를 모름 || DAgger | 같음, **학생이 가는 상태에서** | 전문가를 계속 질의 | 전문가보다 잘할 수 없음 || SAC (warm) | 보상 | 위의 출발점 + 상호작용 | 보상을 잘못 쓰면 그걸 충실히 달성 || SAC (cold) | 보상 | 상호작용 **아주 많이** | 어려운 과제를 시작조차 못 함 |마지막 줄이 융합의 이유다. 그리고 `rl_yeen/reward_ablation.ipynb` 가 세 번째 줄의 위험을따로 다룬다 — `trap` 보상에서는 에이전트가 **일부러 실패하는 쪽이 이득**이다.### 결과물 저장

In [ ]:
import shutilfrom pathlib import Pathout = Path("/content/fusion_results"); out.mkdir(exist_ok=True)for f in Path("pipeline_yeen/checkpoints").glob("*.npz"):    shutil.copy(f, out / f.name)for f in Path("pipeline_yeen").glob("*.csv"):    shutil.copy(f, out / f.name)shutil.make_archive("/content/fusion_results", "zip", out)print(*sorted(p.name for p in out.iterdir()), sep="\n")

In [ ]:
from google.colab import filesfiles.download("/content/fusion_results.zip")